In [ ]:
# ============================================================
# CELL 1: Install RAPIDS (cuDF, cuGraph) for GPU Acceleration
# ============================================================
# Cài đặt bộ thư viện của NVIDIA, bỏ qua các thư viện mạng thuần CPU
!pip install -q cudf-cu12 cugraph-cu12 --extra-index-url=https://pypi.nvidia.com
!pip install -q tqdm pyarrow

import importlib
# Kiểm tra xem GPU và thư viện đã sẵn sàng chưa
for pkg in ["cudf", "cugraph"]:
    assert importlib.util.find_spec(pkg), f"Missing: {pkg}. Lỗi: Bạn chưa bật T4 GPU hoặc cài đặt thất bại."
    
print("✅ All RAPIDS dependencies installed. GPU is ready.")

In [ ]:
# ============================================================
# CELL 2: Configuration + Load data to GPU using cuDF
# ============================================================
import os, gc
import cudf  # THAY THẾ HOÀN TOÀN pandas
import cugraph
from google.colab import drive

drive.mount("/content/drive")

# ─── USER CONFIG ─────────────────────────────────────────────
DRIVE_PATH = "/content/drive/MyDrive/AML/dataset/tx_log.csv"

WINDOW_SIZE   = 5      # number of steps per snapshot
STRIDE        = 2      # step stride between windows
ALERT_THRESH  = 0.30   # community alert_ratio threshold
MIN_COMM_SIZE = 3      # discard communities smaller than this
JACCARD_THRESH = 0.25  # minimum Jaccard overlap
TOP_N_DISPLAY  = 10    # top suspicious communities
# ─────────────────────────────────────────────────────────────

# ─── LOAD (GPU ACCELERATED) ──────────────────────────────────
print("Đang tải dữ liệu trực tiếp vào GPU VRAM...")

if os.path.isdir(DRIVE_PATH):
    # cuDF đọc parquet cực kỳ nhanh
    df = cudf.read_parquet(DRIVE_PATH + "/*.parquet")
    print(f"  Đã tải thư mục parquet.")
else:
    # cuDF đọc CSV bằng nhiều luồng trên GPU
    df = cudf.read_csv(DRIVE_PATH)
    print(f"  Đã tải file CSV.")

# ─── NORMALISE COLUMN NAMES ──────────────────────────────────
col_map = {}
for col in df.columns:
    cl = col.lower().replace(" ", "_").replace(".", "_")
    if   "time"    in cl or "step" in cl:   col_map[col] = "step"
    elif cl in ("from", "account", "sender", "from_id", "nameorig"):
                                            col_map[col] = "src"
    elif cl in ("to", "account_1", "receiver", "to_id", "namedest"):
                                            col_map[col] = "dst"
    elif "amount"  in cl:                   col_map[col] = "amount"
    elif "laundering" in cl or "label" in cl or "fraud" in cl or "issar" in cl:
                                            col_map[col] = "is_laundering"

df = df.rename(columns=col_map)

required = {"step", "src", "dst", "amount", "is_laundering"}
missing = required - set(df.columns)
assert not missing, f"Could not map columns: {missing}\n  Available: {list(df.columns)}"

# ─── RUTHLESS TYPE CASTING (VRAM SURVIVAL) ───────────────────
print("Ép kiểu dữ liệu để cứu sống VRAM...")

df["step"]          = df["step"].astype('int32')
# Giảm một nửa RAM cho số tiền, độ chính xác vẫn thừa đủ cho AML
df["amount"]        = df["amount"].astype('float32')
# Nhãn chỉ có 0 và 1, dùng 1 byte thay vì 8 bytes
df["is_laundering"] = df["is_laundering"].astype('int8')

# KHÔNG DÙNG STRING. Mã hóa tài khoản thành ID số nội bộ.
# cuGraph sau này sẽ tự động mapping lại khi build đồ thị,
# nhưng ép category ở đây giúp dọn dẹp bộ nhớ dataframe ngay lập tức.
df["src"] = df["src"].astype('category')
df["dst"] = df["dst"].astype('category')

# ─── SUMMARY ─────────────────────────────────────────────────
# Mọi phép toán dưới đây đều chạy trên GPU
n_steps = df["step"].nunique()
n_alert = df["is_laundering"].sum()

# Đếm số node duy nhất gộp cả src và dst (concat cuDF)
unique_nodes = cudf.concat([df["src"].astype('int32'), df["dst"].astype('int32')]).nunique()

print(f"\nDataset summary (On GPU)")
print(f"  Rows        : {len(df):>12,}")
print(f"  Unique steps: {n_steps:>11,}")
print(f"  Unique nodes: {unique_nodes:>11,}")
print(f"  Alert edges : {n_alert:>11,}  ({(n_alert/len(df))*100:.2f}%)")
print(f"  Step range  : {df['step'].min()} – {df['step'].max()}")

# Xóa rác hệ thống để lấy lại từng MB VRAM
gc.collect()
print("\n✅ Dữ liệu đã sẵn sàng trên VRAM.")

In [ ]:
# ============================================================
# CELL 3: Build temporal snapshots (GPU Accelerated with RAPIDS)
# ============================================================
import cugraph
import cudf
import cupy as cp
from tqdm.auto import tqdm
import gc

def build_snapshot_gpu(df_window: cudf.DataFrame):
    """
    Xây dựng đồ thị có hướng và tổng hợp đặc trưng trên GPU.
    Thay vì nhét thuộc tính vào object đồ thị, ta trả về:
    1. Đồ thị thuần túy (phục vụ thuật toán Leiden/Louvain).
    2. Bảng đặc trưng Cạnh (Edge Features).
    3. Bảng đặc trưng Nút (Node Features).
    """

    # ── 1. TỔNG HỢP CẠNH (EDGE AGGREGATION) SIÊU TỐC TRÊN GPU ──
    # Use named aggregation to avoid MultiIndex columns
    agg_edges = df_window.groupby(["src", "dst"], as_index=False).agg(
        total_amount=("amount", "sum"),
        n_tx=("amount", "count"),
        n_alert=("is_laundering", "sum")
    )
    
    # Ensure all numeric columns are proper types before passing to graph
    agg_edges["total_amount"] = agg_edges["total_amount"].astype('float32')
    agg_edges["n_tx"] = agg_edges["n_tx"].astype('int32')
    agg_edges["n_alert"] = agg_edges["n_alert"].astype('int32')
    
    agg_edges["alert_ratio"] = agg_edges["n_alert"].astype('float32') / agg_edges["n_tx"].astype('float32')

    # ── 2. XÂY DỰNG ĐỒ THỊ CUGRAPH ─────────────────────────────
    G = cugraph.Graph(directed=True)

    # Convert src and dst columns to their underlying integer codes (int32)
    # before passing to cugraph.Graph.from_cudf_edgelist.
    # This resolves the TypeError: unhashable type: 'CategoricalDtype'
    # that occurs during cugraph's internal renumbering process.
    agg_edges_for_graph = agg_edges.copy() # Create a copy to not alter agg_edges if needed elsewhere
    agg_edges_for_graph["src"] = agg_edges_for_graph["src"].cat.codes.astype('int32')
    agg_edges_for_graph["dst"] = agg_edges_for_graph["dst"].cat.codes.astype('int32')

    G.from_cudf_edgelist(
        agg_edges_for_graph,
        source="src",
        destination="dst",
        edge_attr="total_amount",
        renumber=True
    )

    # ── 3. TỔNG HỢP ĐẶC TRƯNG NÚT (NODE FEATURES) BẰNG VECTOR ──
    # To maintain consistency with the graph's node representation (which is now based on int32 codes),
    # aggregate node features using these same integer codes.
    df_window_with_codes = df_window.copy()
    df_window_with_codes["src_code"] = df_window_with_codes["src"].cat.codes.astype('int32')
    df_window_with_codes["dst_code"] = df_window_with_codes["dst"].cat.codes.astype('int32')

    node_stats_src = df_window_with_codes.groupby("src_code", as_index=False).agg(
        total_volume_src=("amount", "sum"),
        n_tx_src=("amount", "count"),
        n_alert_tx_src=("is_laundering", "sum")
    ).rename(columns={"src_code": "node"})

    node_stats_dst = df_window_with_codes.groupby("dst_code", as_index=False).agg(
        total_volume_dst=("amount", "sum"),
        n_tx_dst=("amount", "count"),
        n_alert_tx_dst=("is_laundering", "sum")
    ).rename(columns={"dst_code": "node"})

    node_features = node_stats_src.merge(node_stats_dst, on="node", how="outer").fillna(0)

    node_features["total_volume"] = node_features["total_volume_src"] + node_features["total_volume_dst"]
    node_features["n_tx"]         = node_features["n_tx_src"] + node_features["n_tx_dst"]
    node_features["n_alert_tx"]   = node_features["n_alert_tx_src"] + node_features["n_alert_tx_dst"]

    node_features["alert_rate"] = node_features["n_alert_tx"] / node_features["n_tx"]
    node_features = node_features[["node", "total_volume", "n_tx", "n_alert_tx", "alert_rate"]]

    return G, agg_edges, node_features

def build_all_snapshots_gpu(df: cudf.DataFrame, window_size: int, stride: int) -> list[dict]:
    # Kéo mảng step về CPU chỉ để xử lý logic cửa sổ (vì nó rất nhỏ)
    steps = df["step"].unique().to_pandas().sort_values().tolist()
    snapshots = []

    starts = range(0, len(steps) - window_size + 1, stride)

    for i in tqdm(starts, desc="Building snapshots (GPU)"):
        window_steps = steps[i : i + window_size]

        # Masking trên GPU cực kỳ nhanh
        mask = df["step"].isin(window_steps)
        df_window = df[mask]

        if len(df_window) == 0:
            continue

        G, edge_feat, node_feat = build_snapshot_gpu(df_window)

        if G.number_of_nodes() == 0:
            continue

        snapshots.append({
            "window_id" : i // stride,
            "steps"     : window_steps,
            "step_start": window_steps[0],
            "step_end"  : window_steps[-1],
            "graph"     : G,
            "edge_feat" : edge_feat,  # Bảng lưu thuộc tính cạnh
            "node_feat" : node_feat,  # Bảng lưu thuộc tính nút
            "n_edges"   : G.number_of_edges(),
            "n_nodes"   : G.number_of_nodes(),
        })

    print(f"\n✅ Built {len(snapshots)} snapshots using RAPIDS "
          f"(window={window_size} steps, stride={stride})")
    return snapshots

# Chạy thử
snapshots = build_all_snapshots_gpu(df, WINDOW_SIZE, STRIDE)

# Kiểm tra sức khỏe nhanh
if len(snapshots) > 0:
    s = snapshots[0]
    print(f"\nSnapshot 0  steps {s['steps'][0]}–{s['steps'][-1]}")
    print(f"  Nodes: {s['n_nodes']:,}   Edges: {s['n_edges']:,}")
    print(s['node_feat'].head()) # Xem thử dữ liệu nút

# Giải phóng RAM/VRAM
gc.collect()


In [ ]:
# ============================================================
# CELL 4: Leiden per snapshot + cross-window tracking (GPU RAPIDS)
# ============================================================
import cugraph
import cudf
from tqdm.auto import tqdm
import gc

def track_communities_gpu(snapshots: list[dict], jaccard_thresh: float = JACCARD_THRESH):
    """
    Chạy thuật toán Leiden trên mỗi snapshot để tìm cộng đồng.
    Theo dõi cộng đồng qua các cửa sổ bằng Jaccard similarity.
    Gán global_cid (ID cộng đồng toàn cục) cho tất cả cộng đồng.
    """
    global_cid_counter = 0
    timeline = []

    for snap_idx, snap in enumerate(tqdm(snapshots, desc="Leiden + Community Tracking")):
        G = snap["graph"]
        edge_df = snap["edge_feat"]    # ['src', 'dst', 'total_amount', 'n_tx', 'n_alert']

        # ── 1. EXTRACT EDGE LIST VÀ SYMMETRIZE ────────────────────────
        _src, _dst, _weights = G.view_edge_list()

        # Tạo DataFrame từ edge list để dễ quản lý
        directed_edgelist = cudf.DataFrame({
            'src': cudf.Series(_src),
            'dst': cudf.Series(_dst),
            'weights': cudf.Series(_weights)
        })

        if len(directed_edgelist) == 0:
            # Graph rỗng, skip
            snap["partition_df"] = cudf.DataFrame(columns=["node", "global_cid"])
            continue

        # check if directed or undirected
        is_directed = G.is_directed()

        if not is_directed:
            # Already undirected, use directly
            G_to_leiden = G
            leiden_weights = 'total_amount'  # edge attribute name from graph
        else:
            # Create reverse edges by swapping src and dst
            reverse_edgelist = directed_edgelist.rename(columns={'src': 'dst', 'dst': 'src'})

            # Concatenate original and reverse edges
            symmetrized_edgelist = cudf.concat([directed_edgelist, reverse_edgelist], ignore_index=True)

            # Ensure 'weights' column is numeric before summing
            symmetrized_edgelist['weights'] = symmetrized_edgelist['weights'].astype('float32')

            # Aggregate weights for undirected edges: for any pair (u,v) and (v,u),
            # sum their weights to get a single combined_weights for the undirected edge.
            undirected_edgelist_aggregated = symmetrized_edgelist.groupby(['src', 'dst'], as_index=False)['weights'].sum()
            undirected_edgelist_aggregated = undirected_edgelist_aggregated.rename(columns={'weights': 'combined_weights'})

            # Build the undirected graph directly from the symmetrized and aggregated edge list
            # Use renumber=True to let cugraph handle node ID conversion automatically
            G_to_leiden = cugraph.Graph(directed=False)
            G_to_leiden.from_cudf_edgelist(
                undirected_edgelist_aggregated,
                source="src",
                destination="dst",
                edge_attr="combined_weights",
                renumber=True  # Let cugraph handle node ID management
            )
            leiden_weights = 'combined_weights'

        # ── 2. RUN LEIDEN ALGORITHM ───────────────────────────────────
        partitions_df, _ = cugraph.leiden(G_to_leiden, weights=leiden_weights)
        partitions_df = partitions_df.rename(columns={'vertex': 'node'})

        if len(partitions_df) == 0:
            snap["partition_df"] = cudf.DataFrame(columns=["node", "global_cid"])
            continue

        # ── 3. CROSS-WINDOW COMMUNITY TRACKING (Jaccard Matching) ─────
        if snap_idx == 0:
            # First window: assign new global_cid to each local community
            local_cids = partitions_df["partition"].unique()

            for local_cid in local_cids:
                partitions_df.loc[partitions_df["partition"] == local_cid, "global_cid"] = global_cid_counter
                global_cid_counter += 1
        else:
            # Subsequent windows: match with previous communities
            prev_snapshot = snapshots[snap_idx - 1]
            prev_partition = prev_snapshot["partition_df"]

            # Get all unique communities in current window
            curr_local_cids = partitions_df["partition"].unique().to_pandas().tolist()
            partitions_df["global_cid"] = -1  # Initialize unknown

            for curr_local_cid in curr_local_cids:
                curr_nodes = partitions_df[partitions_df["partition"] == curr_local_cid]["node"]

                # Find best matching previous community via Jaccard similarity
                if len(curr_nodes) == 0:
                    continue

                # Inner join to find nodes that exist in both windows
                merged = curr_nodes.to_frame().merge(
                    prev_partition[["node", "global_cid"]].rename(columns={"global_cid": "prev_global_cid"}),
                    on="node",
                    how="inner"
                )

                if len(merged) == 0:
                    # No overlap with any previous community -> new community
                    partitions_df.loc[partitions_df["partition"] == curr_local_cid, "global_cid"] = global_cid_counter
                    global_cid_counter += 1
                else:
                    # Find previous community with highest Jaccard overlap
                    prev_cids_in_overlap = merged["prev_global_cid"].unique()
                    best_prev_cid = None
                    best_jaccard = 0

                    for prev_global_cid in prev_cids_in_overlap:
                        overlap_count = len(merged[merged["prev_global_cid"] == prev_global_cid])

                        # Get size of previous community
                        prev_size = len(prev_partition[prev_partition["global_cid"] == prev_global_cid])
                        curr_size = len(curr_nodes)

                        jaccard = overlap_count / (curr_size + prev_size - overlap_count + 1e-9)

                        if jaccard > best_jaccard and jaccard >= jaccard_thresh:
                            best_jaccard = jaccard
                            best_prev_cid = prev_global_cid

                    if best_prev_cid is not None:
                        # Assign best previous community ID
                        partitions_df.loc[partitions_df["partition"] == curr_local_cid, "global_cid"] = best_prev_cid
                    else:
                        # No strong match -> assign new global_cid
                        partitions_df.loc[partitions_df["partition"] == curr_local_cid, "global_cid"] = global_cid_counter
                        global_cid_counter += 1

        # ── 4. STORE PARTITION MAPPING ────────────────────────────────
        snap["partition_df"] = partitions_df[["node", "global_cid"]].copy()
        timeline.append((snap_idx, len(partitions_df), partitions_df["global_cid"].nunique()))

    # Print summary
    print(f"\n✅ Leiden community detection + tracking complete")
    print(f"   Total unique global communities: {global_cid_counter}")
    print(f"   Timeline (window_idx, n_nodes, n_communities):")
    for w_idx, n_nodes, n_comms in timeline[:5]:
        print(f"     Window {w_idx}: {n_nodes:,} nodes, {n_comms} communities")
    if len(timeline) > 5:
        print(f"     ... ({len(timeline) - 5} more windows)")

    gc.collect()
    return snapshots, global_cid_counter

# Chạy Leiden algorithm và community tracking
snapshots, N_GLOBAL_COMMS = track_communities_gpu(snapshots, JACCARD_THRESH)


In [ ]:
# ============================================================
# CELL 5: Community feature engineering + suspicious labelling (GPU)
# ============================================================
import cudf
import cupy as cp
from tqdm.auto import tqdm
import gc

def extract_features_gpu(snapshots: list[dict], df_raw: cudf.DataFrame, min_comm_size: int = MIN_COMM_SIZE):
    """
    Trích xuất đặc trưng cộng đồng không dùng Vòng Lặp nội bộ.
    Tất cả được tính toán bằng các phép Join và Groupby trên VRAM.
    """
    comm_records_list = []

    for snap in tqdm(snapshots, desc="GPU Feature Engineering"):
        edge_df = snap["edge_feat"]      # ['src', 'dst', 'total_amount', 'n_tx', 'n_alert']
        part_df = snap["partition_df"]   # ['node', 'global_cid']
        
        if len(part_df) == 0:
            continue

        # ── 1. TÌM CẠNH NỘI BỘ BẰNG INNER JOIN (Thay thế List Comprehension) ──
        # Rename partition_df columns for clarity in joins
        part_df_src = part_df.rename(columns={'node': 'src', 'global_cid': 'src_cid'})
        part_df_dst = part_df.rename(columns={'node': 'dst', 'global_cid': 'dst_cid'})
        
        # Ánh xạ src thành global_cid
        edges_mapped = edge_df.merge(part_df_src, on="src", how="inner")
        
        # Ánh xạ dst thành global_cid
        edges_mapped = edges_mapped.merge(part_df_dst, on="dst", how="inner")

        # Lọc cạnh nội bộ: Khi src và dst thuộc cùng một cộng đồng
        internal_edges = edges_mapped[edges_mapped["src_cid"] == edges_mapped["dst_cid"]].copy()
        internal_edges = internal_edges.rename(columns={"src_cid": "global_cid"}).drop(columns=["dst_cid"])
        
        if len(internal_edges) == 0:
            continue

        # ── 2. TÍNH ĐẶC TRƯNG CƠ BẢN (Vectorized Aggregation) ──
        internal_edges["is_alert_edge"] = (internal_edges["n_alert"] > 0).astype("float32")
        
        # Use named aggregation to avoid MultiIndex columns
        comm_stats = internal_edges.groupby("global_cid", as_index=False).agg(
            total_volume=("total_amount", "sum"),
            n_internal_edges=("total_amount", "count"),
            alert_ratio=("is_alert_edge", "mean")
        )

        # ── 3. TÍNH CHỈ SỐ HHI (Thay cho Gini - Đo lường sự tập trung dòng tiền) ──
        internal_edges["amount_sq"] = internal_edges["total_amount"] ** 2
        hhi_stats = internal_edges.groupby("global_cid", as_index=False).agg(sum_amount_sq=("amount_sq", "sum"))
        
        comm_stats = comm_stats.merge(hhi_stats, on="global_cid")
        # HHI = Sum(x^2) / (Sum(x))^2. HHI tiến về 1 nghĩa là tiền dồn hết vào 1 cạnh (Layering/Sink).
        comm_stats["flow_concentration"] = comm_stats["sum_amount_sq"] / (comm_stats["total_volume"] ** 2 + 1e-9)
        comm_stats = comm_stats.drop(columns=["sum_amount_sq"])

        # ── 4. TÍNH SINK RATIO (Tỷ lệ dồn tiền về một nút đích lớn nhất) ──
        # Tính in_degree volume cho mỗi nút đích trong cộng đồng
        node_in_volume = internal_edges.groupby(["global_cid", "dst"], as_index=False).agg(in_vol=("total_amount", "sum"))
        # Lấy nút nhận được nhiều tiền nhất đại diện cho nhóm
        max_in_vol = node_in_volume.groupby("global_cid", as_index=False).agg(max_in=("in_vol", "max"))
        
        comm_stats = comm_stats.merge(max_in_vol, on="global_cid", how="left")
        comm_stats["sink_ratio"] = comm_stats["max_in"] / (comm_stats["total_volume"] + 1e-9)
        comm_stats = comm_stats.drop(columns=["max_in"])

        # ── 5. KẾT HỢP KÍCH THƯỚC VÀ LỌC ──
        sizes = part_df.groupby("global_cid", as_index=False).agg(size=("node", "count"))
        comm_stats = comm_stats.merge(sizes, on="global_cid")
        
        # Loại bỏ các cộng đồng quá nhỏ
        comm_stats = comm_stats[comm_stats["size"] >= min_comm_size]

        # ── 6. LƯU METADATA CỦA SNAPSHOT ──
        comm_stats["window_id"] = snap["window_id"]
        comm_stats["step_start"] = snap["step_start"]
        comm_stats["step_end"] = snap["step_end"]

        comm_records_list.append(comm_stats)

    # Nối toàn bộ kết quả thành một DataFrame duy nhất trên GPU
    if comm_records_list:
        final_comm_df = cudf.concat(comm_records_list, ignore_index=True)
    else:
        final_comm_df = cudf.DataFrame()
        
    return final_comm_df

# Chạy pipeline trích xuất
comm_df = extract_features_gpu(snapshots, df)

# ─── GÁN NHÃN ĐÁNG NGỜ (SUSPICIOUS LABELING) ────────────────
if len(comm_df) > 0:
    comm_df["is_suspicious"] = (comm_df["alert_ratio"] >= ALERT_THRESH).astype('int8')

    # Chuyển về Pandas một lần duy nhất ở bước cuối để in báo cáo
    comm_df_pd = comm_df.to_pandas()
    
    total = len(comm_df_pd)
    susp  = comm_df_pd["is_suspicious"].sum()
    
    print(f"\n✅ Community feature table generated: {total:,} entries")
    print(f"   Suspicious (alert_ratio ≥ {ALERT_THRESH}): {susp:,}  ({susp/total*100:.1f}%)")
    print(f"\nFeature Statistics:")
    print(comm_df_pd[["size", "alert_ratio", "flow_concentration", "sink_ratio"]].describe().round(3))
else:
    print("\nKhông tìm thấy cộng đồng nào thỏa mãn điều kiện kích thước.")

gc.collect()


In [ ]:
# ============================================================
# CELL 6: Evaluation + visualisation
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

# ─── Convert to Pandas for analysis ──────────────────────────
comm_df_pd = comm_df.to_pandas()
df_pd = df.to_pandas()

# ─── 1. Alert Enrichment Ratio ────────────────────────────────
global_alert_rate = df_pd["is_laundering"].mean()
susp_alert_rate   = comm_df_pd[comm_df_pd["is_suspicious"] == 1]["alert_ratio"].mean()
norm_alert_rate   = comm_df_pd[comm_df_pd["is_suspicious"] == 0]["alert_ratio"].mean()

AER = susp_alert_rate / (global_alert_rate + 1e-9)

print("=" * 55)
print(f"  ALERT ENRICHMENT RATIO (AER)    : {AER:.2f}×")
print(f"  Global edge alert rate          : {global_alert_rate*100:.2f}%")
print(f"  Suspicious community alert rate : {susp_alert_rate*100:.2f}%")
print(f"  Normal community alert rate     : {norm_alert_rate*100:.2f}%")
print("=" * 55)

# ─── 2. Plots ────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

palette = {0: "#5DCAA5", 1: "#E24B4A"}

# 2a: alert_ratio distribution
ax0 = fig.add_subplot(gs[0, 0])
for label, grp in comm_df_pd.groupby("is_suspicious"):
    ax0.hist(grp["alert_ratio"], bins=30, alpha=0.7,
             color=palette[label], label=f"{'Suspicious' if label else 'Normal'}")
ax0.axvline(ALERT_THRESH, ls="--", color="black", lw=1, label=f"Threshold={ALERT_THRESH}")
ax0.set_title("Alert ratio distribution")
ax0.set_xlabel("alert_ratio")
ax0.legend(fontsize=8)

# 2b: flow_concentration vs alert_ratio
ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(
    comm_df_pd["flow_concentration"],
    comm_df_pd["alert_ratio"],
    c=comm_df_pd["is_suspicious"].map(palette),
    alpha=0.4, s=8, linewidths=0,
)
ax1.set_title("Flow concentration vs alert ratio")
ax1.set_xlabel("flow_concentration (HHI)")
ax1.set_ylabel("alert_ratio")

# 2c: sink_ratio distribution
ax2 = fig.add_subplot(gs[0, 2])
for label, grp in comm_df_pd.groupby("is_suspicious"):
    ax2.hist(grp["sink_ratio"].clip(upper=1.0), bins=30, alpha=0.7,
             color=palette[label], label=f"{'Suspicious' if label else 'Normal'}")
ax2.set_title("Sink ratio distribution")
ax2.set_xlabel("sink_ratio (clipped at 1.0)")
ax2.legend(fontsize=8)

# 2d: community size vs alert_ratio
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(
    comm_df_pd["size"],
    comm_df_pd["alert_ratio"],
    c=comm_df_pd["is_suspicious"].map(palette),
    alpha=0.4, s=8, linewidths=0,
)
ax3.set_title("Community size vs alert ratio")
ax3.set_xlabel("Community size (nodes)")
ax3.set_ylabel("alert_ratio")
ax3.set_xscale('log')

# 2e: suspicious community count over windows
ax4 = fig.add_subplot(gs[1, 1])
win_counts = comm_df_pd.groupby(["window_id", "is_suspicious"]).size().unstack(fill_value=0)
ax4.stackplot(
    win_counts.index,
    win_counts.get(0, pd.Series(dtype=int)),
    win_counts.get(1, pd.Series(dtype=int)),
    labels=["Normal", "Suspicious"],
    colors=["#5DCAA5", "#E24B4A"],
    alpha=0.75,
)
ax4.set_title("Community count per window")
ax4.set_xlabel("Window ID")
ax4.set_ylabel("# communities")
ax4.legend(fontsize=8, loc="upper right")

# 2f: top suspicious communities by total_volume
ax5 = fig.add_subplot(gs[1, 2])
top_susp = (
    comm_df_pd[comm_df_pd["is_suspicious"] == 1]
    .groupby("global_cid")["total_volume"].sum()
    .nlargest(TOP_N_DISPLAY)
    .sort_values()
)
if len(top_susp) > 0:
    ax5.barh(top_susp.index.astype(str), top_susp.values, color="#E24B4A", alpha=0.8)
    ax5.set_title(f"Top {TOP_N_DISPLAY} suspicious communities\n(total transaction volume)")
    ax5.set_xlabel("Total volume")
    ax5.set_ylabel("Global community ID")
else:
    ax5.text(0.5, 0.5, "No suspicious communities found", ha='center', va='center')
    ax5.set_title("Top suspicious communities")

plt.suptitle("AML Community Detection — Evaluation Dashboard", fontsize=13, y=1.01)
plt.savefig("/content/community_detection_dashboard.png", bbox_inches="tight", dpi=150)
plt.show()
print("✅ Dashboard saved to /content/community_detection_dashboard.png")

# ─── 3. Summary Statistics ──────────────────────────────────
print(f"\n✅ Feature Statistics (from {len(comm_df_pd):,} communities):")
print(comm_df_pd[["size", "alert_ratio", "flow_concentration", "sink_ratio"]].describe().round(3))

In [ ]:
# ============================================================
# CELL 7: Export artefacts for GNN, motif mining, GAT attention
# ============================================================
import pickle
import json

OUTPUT_DIR = "/content/drive/MyDrive/AML_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── 7a: Community feature table (GPU → CPU) ─────────────────
feat_path = os.path.join(OUTPUT_DIR, "community_features.csv")
comm_df_export = comm_df_pd[["global_cid", "window_id", "step_start", "step_end", 
                              "size", "n_internal_edges", "total_volume", "alert_ratio",
                              "flow_concentration", "sink_ratio", "is_suspicious"]]
comm_df_export.to_csv(feat_path, index=False)
print(f"✅ community_features.csv  → {feat_path}")
print(f"   ({len(comm_df_export):,} rows)")

# ─── 7b: Node → community mapping (all windows) ──────────────
# Build mapping from partition_df in each snapshot
records = []
for snap in snapshots:
    partition_df = snap["partition_df"].to_pandas() if hasattr(snap["partition_df"], "to_pandas") else snap["partition_df"]
    
    for _, row in partition_df.iterrows():
        records.append({
            "node"      : int(row["node"]),
            "window_id" : snap["window_id"],
            "step_start": snap["step_start"],
            "step_end"  : snap["step_end"],
            "global_cid": int(row["global_cid"]),
        })

node_map_df = pd.DataFrame(records)

# Merge suspicious label from community features
label_map = comm_df_export[["global_cid", "window_id", "is_suspicious"]].drop_duplicates()
node_map_df = node_map_df.merge(label_map, on=["global_cid", "window_id"], how="left")
node_map_df["is_suspicious"] = node_map_df["is_suspicious"].fillna(0).astype(int)

map_path = os.path.join(OUTPUT_DIR, "node_community_map.csv")
node_map_df.to_csv(map_path, index=False)
print(f"✅ node_community_map.csv  → {map_path}")
print(f"   ({len(node_map_df):,} rows)")

# ─── 7c: Top-K suspicious community metadata (JSON) ──────────
# Export metadata for top suspicious communities without attempting subgraph visualization
TOP_K_EXPORT = min(50, len(comm_df_export[comm_df_export["is_suspicious"] == 1]))

top_suspicious_records = (
    comm_df_export[comm_df_export["is_suspicious"] == 1]
    .groupby("global_cid")["alert_ratio"]
    .mean()
    .nlargest(TOP_K_EXPORT)
    .index.tolist()
)

subgraph_metadata = {}
for gcid in top_suspicious_records:
    rows = comm_df_export[comm_df_export["global_cid"] == gcid].sort_values(
        "total_volume", ascending=False
    )
    if len(rows) > 0:
        best = rows.iloc[0]
        window_id = int(best["window_id"])
        snap = snapshots[window_id]
        partition_df = snap["partition_df"].to_pandas() if hasattr(snap["partition_df"], "to_pandas") else snap["partition_df"]
        
        # Get member nodes for this community in this window
        members = partition_df[partition_df["global_cid"] == gcid]["node"].tolist()
        
        subgraph_metadata[int(gcid)] = {
            "global_cid"       : int(gcid),
            "window_id"        : int(best["window_id"]),
            "step_start"       : int(best["step_start"]),
            "step_end"         : int(best["step_end"]),
            "n_members"        : int(best["size"]),
            "n_edges"          : int(best["n_internal_edges"]),
            "total_volume"     : float(best["total_volume"]),
            "alert_ratio"      : float(best["alert_ratio"]),
            "flow_concentration": float(best["flow_concentration"]),
            "sink_ratio"       : float(best["sink_ratio"]),
            "member_nodes"     : members,  # List of node IDs in this community
        }

meta_path = os.path.join(OUTPUT_DIR, "top_suspicious_communities.json")
with open(meta_path, "w") as f:
    json.dump(subgraph_metadata, f, indent=2)
print(f"✅ top_suspicious_communities.json → {meta_path}")
print(f"   ({len(subgraph_metadata)} communities)")

# ─── 7d: Summary report ──────────────────────────────────────
report = {
    "execution_summary": {
        "n_windows"         : len(snapshots),
        "n_global_communities": int(N_GLOBAL_COMMS),
        "n_suspicious_communities": int(comm_df_export["is_suspicious"].sum()),
        "n_total_nodes"     : int(node_map_df["node"].nunique()),
    },
    "metrics": {
        "alert_enrichment_ratio": float(AER),
        "global_alert_rate" : float(global_alert_rate),
        "susp_alert_rate"   : float(susp_alert_rate),
        "normal_alert_rate" : float(norm_alert_rate),
    },
    "configuration": {
        "window_size"       : WINDOW_SIZE,
        "stride"            : STRIDE,
        "alert_threshold"   : ALERT_THRESH,
        "jaccard_threshold" : JACCARD_THRESH,
        "min_community_size": MIN_COMM_SIZE,
    },
}

report_path = os.path.join(OUTPUT_DIR, "run_summary.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"\n✅ run_summary.json        → {report_path}")
print(f"\n{'='*55}")
print(f"  EXECUTION SUMMARY")
print(f"{'='*55}")
for category, values in report.items():
    print(f"\n  {category.upper()}:")
    for k, v in values.items():
        print(f"    {k:<28}: {v}")

print(f"\n{'='*55}")
print(f"  All outputs written to: {OUTPUT_DIR}")
print(f"{'='*55}")